# as-strided-noncontig-source — worked example 3: Extract the anti-diagonal with stride arithmetic

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `as-strided-noncontig-source`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

Stride arithmetic can pick out non-obvious diagonals. For a square `(N, N)` matrix with stride `(sR, sC)`, the *main* diagonal advances `sR + sC` per step. The **anti-diagonal** (top-right to bottom-left) advances one row down but one column *left*, i.e. `sR - sC` per step, starting from element `[0, N-1]`.

## Worked solution

Goal: a zero-copy 1-D view of the anti-diagonal `[m[0,N-1], m[1,N-2], ..., m[N-1,0]]` using only `as_strided`.

1. `as_strided(input, size, stride, storage_offset)` reads from `input.storage()` starting at `storage_offset` (in elements). We must start at `m[0, N-1]`.
2. The starting offset, for a contiguous matrix with stride `(sR, sC)`, is `0*sR + (N-1)*sC = (N-1)*sC`.
3. Each step along the anti-diagonal moves +1 row (`+sR`) and -1 column (`-sC`), so the single-axis stride is `sR - sC`.
4. The shape is `(N,)`. Call `t.as_strided(m, size=(N,), stride=(sR - sC,), storage_offset=(N - 1) * sC)`.
5. We verify against the reference `t.flip(m, dims=[1]).diagonal()` — flipping columns turns the anti-diagonal into the main diagonal, whose values must match.
6. Printing the view shows the anti-diagonal entries; `data_ptr` confirms it aliases `m`.

In [ ]:
def antidiagonal_via_strided(m: Tensor) -> Tensor:
    N = m.shape[0]
    sR, sC = m.stride()
    return t.as_strided(m, size=(N,), stride=(sR - sC,), storage_offset=(N - 1) * sC)

m = t.arange(16, dtype=t.float32).reshape(4, 4)
anti = antidiagonal_via_strided(m)
ref = t.flip(m, dims=[1]).diagonal()
print('anti-diagonal:', anti.tolist())
print('reference    :', ref.tolist())
print('matches      :', t.equal(anti, ref))
print('shares store :', anti.data_ptr() == m.data_ptr())